> **SOLUTION VERSION.** Every exercise is completed. Stuck in the workshop? Copy what you need from here.

# Lab 1: From Prompt to Structured Output
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/solutions/01_prompt_to_structured_output.ipynb)

An LLM predicts text. In this lab you will see three things:
1. What happens when you ask about facts the model does not have
2. How **grounding** (giving it the data) fixes that
3. How **structured output** (JSON) lets your code use the answer

Throughout the workshop we are building the **iGET Conference Concierge**, an assistant that
helps attendees find sessions. The program is sample data in `workshop/data/sessions.json`.

### Setup
Run the cell below. The last line tells you which **brain** is answering:
`openai` or `gemini` (real AI in the cloud), `ollama` (AI on your own computer) or `mock`
(offline practice brain). All of them work for every exercise in this workshop.

In Colab, the first time a notebook reads your key, a pop-up asks for permission:
click **Grant access**. Every lab asks once.

In [ ]:
# SETUP: run this cell first. It is safe to run more than once.
import os, sys, subprocess
REPO_URL = "https://github.com/bmdayal/from-prompt-to-agent"

if os.path.isdir("../workshop"):        # running locally from notebooks/ or solutions/
    sys.path.insert(0, "..")
elif os.path.isdir("workshop"):         # running from the repo folder
    sys.path.insert(0, ".")
else:                                   # running in Google Colab: download the workshop files
    if not os.path.isdir("from-prompt-to-agent"):
        print("Downloading workshop files...")
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    sys.path.insert(0, "from-prompt-to-agent")

import json
from workshop import llm

PROVIDER = "auto"   # "auto", "openai", "gemini", "ollama" or "mock" (offline practice brain)
llm.configure(PROVIDER)

## Experiment 1: Ask without any data
**Predict first:** the model has never seen our conference program. What will it say?
Shout out your guess, then run the cell.

In [ ]:
question = "Which sessions about agents are on Thursday at the iGET 2026 conference?"
print(llm.ask(question))

**Discuss:** Did it admit it does not know, or did it confidently invent sessions, rooms and times?
Confident invention is called a **hallucination**. Remember the role-play: this is the LLM
answering without the Tool card.

## Experiment 2: Grounding, give the model the data
We paste the program into the prompt and tell the model to use **only** that data.

In [ ]:
from workshop.tools import SESSIONS

keys = ["id", "title", "day", "start", "end", "room", "level", "tags"]
program = json.dumps([{k: s[k] for k in keys} for s in SESSIONS])
print(f"The program has {len(SESSIONS)} sessions, {len(program)} characters of text.")

grounded_prompt = f"""Answer using ONLY the sessions data below.
If the answer is not in the data, say you could not find it.

SESSIONS DATA:
{program}

QUESTION: {question}"""

print(llm.ask(grounded_prompt))

Much better. **But** pasting the whole program into every prompt does not scale: a real conference
has hundreds of sessions, every character costs time and money, and some data is private.
Keep that thought. In Lab 2, **tools** solve it.

## Experiment 3: Structured output
People like sentences. **Programs like JSON.** If we ask for JSON, our code can sort, filter
and display the answer, or pass it to another program.

In [ ]:
json_prompt = grounded_prompt + """

Reply ONLY with a JSON list of objects with the keys "id", "title", "start" and "room"."""

raw = llm.ask(json_prompt, json_mode=True)
print("Raw reply from the model:")
print(raw)

sessions = llm.parse_json(raw) or []
if isinstance(sessions, dict):   # some models wrap the list, e.g. {"sessions": [...]}
    sessions = next((v for v in sessions.values() if isinstance(v, list)), [])
print("\nThe same data, used by our code:")
for s in sessions:
    print(f"{s.get('start', '?'):6} | {s.get('room', '?'):15} | {s.get('title', '?')}")

**Why this matters for agents:** when an agent decides "call the search tool with topic = agents",
that decision is also structured output. JSON is the bridge **from prompt to agent**.

## Your turn
Write a prompt that returns **beginner-level sessions on Friday** as JSON with the keys
`"title"` and `"level"`. Use `program` for the data, like the example above.

In [ ]:
my_prompt = f"""Answer using ONLY the sessions data below.

SESSIONS DATA:
{program}

QUESTION: Which Beginner level sessions are on Friday?
Reply ONLY with a JSON list of objects with the keys "title" and "level"."""

raw = llm.ask(my_prompt, json_mode=True)
print(llm.parse_json(raw))

## Key takeaways
- An LLM predicts plausible text. Without data, plausible can mean **made up**.
- **Grounding** (giving the model the facts) makes answers trustworthy.
- **Structured output** (JSON) turns answers into something code can use.

**Next: Lab 2, Giving the Model Tools.** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bmdayal/from-prompt-to-agent/blob/main/solutions/02_tools.ipynb)

Click the badge to open the next lab in a new Colab tab, then run its setup cell first. (Each lab is self-contained, so you can close this one.)